In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, learning_curve, validation_curve, train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,make_scorer
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor
from sklearn.tree import DecisionTreeRegressor
import xgboost as xgb
import lightgbm as lgb
import time
import warnings
warnings.filterwarnings('ignore')

In [ ]:
train = pd.read_csv('/content/sample_data/train_v2.csv', low_memory=False)
store = pd.read_csv('/content/sample_data/store.csv')
print(f" Train data loaded: {train.shape}")
print(f" Store data loaded: {store.shape}")

 Train data loaded: (66900, 9)
 Store data loaded: (1115, 10)


In [ ]:
# RMSPE Function (din cod original)
def rmspe(y_true, y_pred):
    y_true = np.array(y_true)
    y_pred = np.array(y_pred)
    mask = y_true != 0
    return np.sqrt(np.mean(np.square((y_true[mask] - y_pred[mask]) / y_true[mask])))


In [ ]:
# Scorer pentru GridSearch
rmspe_scorer = make_scorer(rmspe, greater_is_better=False)

In [ ]:
# Merge
train = train.merge(store, on='Store', how='left')
print(f"Data merged: {train.shape}")

Data merged: (66900, 18)


In [ ]:
# Remove closed days and zero sales
train = train[train['Open'] == 1]
train = train[train['Sales'] > 0]
print(f" Filtered data: {train.shape}")

 Filtered data: (54688, 18)


In [ ]:
# Convert date and create temporal features
print("\n Creating features...")
train['Date'] = pd.to_datetime(train['Date'])
train['Year'] = train['Date'].dt.year
train['Month'] = train['Date'].dt.month
train['Day'] = train['Date'].dt.day
train['WeekOfYear'] = train['Date'].dt.isocalendar().week
train['DayOfYear'] = train['Date'].dt.dayofyear

# Encoding categorical variables
train['StoreType'] = train['StoreType'].astype('category').cat.codes
train['Assortment'] = train['Assortment'].astype('category').cat.codes
train['StateHoliday'] = train['StateHoliday'].astype(str).replace('0', '0')
train['StateHoliday'] = train['StateHoliday'].astype('category').cat.codes

# Fill missing values
train['CompetitionDistance'].fillna(train['CompetitionDistance'].median(), inplace=True)
train['CompetitionOpenSinceMonth'].fillna(0, inplace=True)
train['CompetitionOpenSinceYear'].fillna(0, inplace=True)
train['Promo2SinceWeek'].fillna(0, inplace=True)
train['Promo2SinceYear'].fillna(0, inplace=True)
train['PromoInterval'].fillna('None', inplace=True)
train['PromoInterval'] = train['PromoInterval'].astype('category').cat.codes

print(" Features created and encoded")


 Creating features...
 Features created and encoded


In [ ]:
# Separate target variable
y_train = train['Sales']
train = train.drop('Sales', axis=1)

In [ ]:
# Select features
feature_cols = ['Store', 'DayOfWeek', 'Promo', 'SchoolHoliday', 'StoreType',
                'Assortment', 'CompetitionDistance', 'Promo2', 'Year', 'Month',
                'Day', 'WeekOfYear', 'DayOfYear', 'StateHoliday',
                'CompetitionOpenSinceMonth', 'CompetitionOpenSinceYear',
                'Promo2SinceWeek', 'Promo2SinceYear', 'PromoInterval']

X_train = train[feature_cols]

# Train-validation split
X_train_split, X_val, y_train_split, y_val = train_test_split(X_train, y_train, test_size=0.2, random_state=42)

print(f"\n Training set: {X_train_split.shape}")
print(f" Validation set: {X_val.shape}")


 Training set: (43750, 19)
 Validation set: (10938, 19)


In [ ]:
def tune_and_visualize_model(model_name, base_model, param_grid, X_train, y_train,
                              X_val, y_val, scaled=False):

    print(f"\n{'='*80}")
    print(f" TUNING {model_name}")
    print(f"{'='*80}")


    if scaled:
        scaler = StandardScaler()
        X_train_proc = scaler.fit_transform(X_train)
        X_val_proc = scaler.transform(X_val)
    else:
        X_train_proc = X_train
        X_val_proc = X_val

    # GridSearch
    n_combinations = np.prod([len(v) for v in param_grid.values()])
    print(f"Testing {n_combinations} combinations...")

    grid_search = GridSearchCV(
        base_model,
        param_grid,
        cv=5,
        scoring=rmspe_scorer,
        return_train_score=True,
        n_jobs=-1,
        verbose=2
    )

    start_time = time.time()
    grid_search.fit(X_train_proc, y_train)
    elapsed = time.time() - start_time

    # Best model prediction pe validation
    best_pred = grid_search.best_estimator_.predict(X_val_proc)
    train_pred = grid_search.best_estimator_.predict(X_train_proc) # Added this line
    val_rmspe = rmspe(y_val, best_pred)
    val_rmse = np.sqrt(mean_squared_error(y_val, best_pred))
    train_mae = mean_absolute_error(y_train, train_pred) # train_pred is now defined
    val_r2 = r2_score(y_val, best_pred)

    print(f"\n Best Parameters:")
    for param, value in grid_search.best_params_.items():
        print(f"   {param}: {value}")
    print(f"\n Results:")
    print(f"   CV RMSPE: {-grid_search.best_score_:.4f}")
    print(f"   Val RMSPE: {val_rmspe:.4f}")
    print(f"   Val RMSE: {val_rmse:.2f}")
    print(f"   Val MAE:  {train_mae:.2f}") # Changed from val_mae to train_mae
    print(f"   Val R²: {val_r2:.4f}")
    print(f"   Time: {elapsed:.2f}s")

    return grid_search.best_estimator_, grid_search.best_params_, val_rmspe

In [ ]:
# Dictionary to store the optimized results
optimized_models = {}
tuning_summary = []

1. Decision Tree

In [ ]:
dt_params = {
    'max_depth': [10, 15, 20, 25, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8]
}

best_dt, params_dt, score_dt = tune_and_visualize_model(
    'Decision Tree',
    DecisionTreeRegressor(random_state=42),
    dt_params,
    X_train_split, y_train_split, X_val, y_val,
    scaled=False
)

optimized_models['Decision Tree'] = {'model': best_dt, 'scaled': False}
tuning_summary.append({
    'Model': 'Decision Tree',
    'Best_Val_RMSPE': score_dt,
    'Best_Params': str(params_dt)
})


 TUNING Decision Tree
Testing 80 combinations...
Fitting 5 folds for each of 80 candidates, totalling 400 fits

 Best Parameters:
   max_depth: 30
   min_samples_leaf: 1
   min_samples_split: 10

 Results:
   CV RMSPE: 0.3315
   Val RMSPE: 0.3163
   Val RMSE: 2179.50
   Val MAE:  1334.19
   Val R²: 0.6041
   Time: 111.92s


2. Random Forest

In [ ]:
rf_params = {
    'n_estimators': [100, 150,200],
    'max_depth': [15, 20],
    'min_samples_split': [5, 10, 15]
}

best_rf, params_rf, score_rf = tune_and_visualize_model(
    'Random Forest',
    RandomForestRegressor(random_state=42,verbose=2, bootstrap=False),
    rf_params,
    X_train_split, y_train_split, X_val, y_val,
    scaled=False
)

optimized_models['Random Forest'] = {'model': best_rf, 'scaled': False}
tuning_summary.append({
    'Model': 'Random Forest',
    'Best_Val_RMSPE': score_rf,
    'Best_Params': str(params_rf)
})


 TUNING Random Forest
Testing 18 combinations...
Fitting 5 folds for each of 18 candidates, totalling 90 fits
building tree 1 of 200
building tree 2 of 200
building tree 3 of 200
building tree 4 of 200
building tree 5 of 200
building tree 6 of 200
building tree 7 of 200
building tree 8 of 200
building tree 9 of 200
building tree 10 of 200
building tree 11 of 200
building tree 12 of 200
building tree 13 of 200
building tree 14 of 200
building tree 15 of 200
building tree 16 of 200
building tree 17 of 200
building tree 18 of 200
building tree 19 of 200
building tree 20 of 200
building tree 21 of 200
building tree 22 of 200
building tree 23 of 200
building tree 24 of 200
building tree 25 of 200
building tree 26 of 200
building tree 27 of 200
building tree 28 of 200
building tree 29 of 200
building tree 30 of 200
building tree 31 of 200
building tree 32 of 200
building tree 33 of 200
building tree 34 of 200
building tree 35 of 200
building tree 36 of 200
building tree 37 of 200
building t

[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:   13.2s


building tree 41 of 200
building tree 42 of 200
building tree 43 of 200
building tree 44 of 200
building tree 45 of 200
building tree 46 of 200
building tree 47 of 200
building tree 48 of 200
building tree 49 of 200
building tree 50 of 200
building tree 51 of 200
building tree 52 of 200
building tree 53 of 200
building tree 54 of 200
building tree 55 of 200
building tree 56 of 200
building tree 57 of 200
building tree 58 of 200
building tree 59 of 200
building tree 60 of 200
building tree 61 of 200
building tree 62 of 200
building tree 63 of 200
building tree 64 of 200
building tree 65 of 200
building tree 66 of 200
building tree 67 of 200
building tree 68 of 200
building tree 69 of 200
building tree 70 of 200
building tree 71 of 200
building tree 72 of 200
building tree 73 of 200
building tree 74 of 200
building tree 75 of 200
building tree 76 of 200
building tree 77 of 200
building tree 78 of 200
building tree 79 of 200
building tree 80 of 200
building tree 81 of 200
building tree 82

[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:   54.0s


building tree 162 of 200
building tree 163 of 200
building tree 164 of 200
building tree 165 of 200
building tree 166 of 200
building tree 167 of 200
building tree 168 of 200
building tree 169 of 200
building tree 170 of 200
building tree 171 of 200
building tree 172 of 200
building tree 173 of 200
building tree 174 of 200
building tree 175 of 200
building tree 176 of 200
building tree 177 of 200
building tree 178 of 200
building tree 179 of 200
building tree 180 of 200
building tree 181 of 200
building tree 182 of 200
building tree 183 of 200
building tree 184 of 200
building tree 185 of 200
building tree 186 of 200
building tree 187 of 200
building tree 188 of 200
building tree 189 of 200
building tree 190 of 200
building tree 191 of 200
building tree 192 of 200
building tree 193 of 200
building tree 194 of 200
building tree 195 of 200
building tree 196 of 200
building tree 197 of 200
building tree 198 of 200
building tree 199 of 200
building tree 200 of 200


[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:  1.1min finished
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.1s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    0.5s
[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    0.6s finished
[Parallel(n_jobs=1)]: Done  40 tasks      | elapsed:    0.4s
[Parallel(n_jobs=1)]: Done 161 tasks      | elapsed:    1.8s



 Best Parameters:
   max_depth: 20
   min_samples_split: 10
   n_estimators: 200

 Results:
   CV RMSPE: 0.3421
   Val RMSPE: 0.3289
   Val RMSE: 2227.56
   Val MAE:  868.24
   Val R²: 0.5865
   Time: 2864.15s


[Parallel(n_jobs=1)]: Done 200 out of 200 | elapsed:    2.2s finished


3.  XGBOOST

In [ ]:
xgb_params = {
    'n_estimators': [100, 150, 200],
    'max_depth': [5, 7, 9],
    'learning_rate': [0.05, 0.1, 0.15],
    'subsample': [0.8, 0.9, 1.0]
}

best_xgb, params_xgb, score_xgb = tune_and_visualize_model(
    'XGBoost',
    xgb.XGBRegressor(random_state=42),
    xgb_params,
    X_train_split, y_train_split, X_val, y_val,
    scaled=False
)

optimized_models['XGBoost'] = {'model': best_xgb, 'scaled': False}
tuning_summary.append({
    'Model': 'XGBoost',
    'Best_Val_RMSPE': score_xgb,
    'Best_Params': str(params_xgb)
})



 TUNING XGBoost
Testing 81 combinations...
Fitting 5 folds for each of 81 candidates, totalling 405 fits

 Best Parameters:
   learning_rate: 0.15
   max_depth: 9
   n_estimators: 200
   subsample: 0.8

 Results:
   CV RMSPE: 0.1512
   Val RMSPE: 0.1452
   Val RMSE: 978.09
   Val MAE:  688.44
   Val R²: 0.9203
   Time: 495.21s


 4. LIGHTGBM

In [ ]:
lgb_params = {
    'n_estimators': [100, 150, 200],
    'max_depth': [5, 7, 9, -1],
    'learning_rate': [0.05, 0.1, 0.15],
    'num_leaves': [31, 50, 70]
}

best_lgb, params_lgb, score_lgb = tune_and_visualize_model(
    'LightGBM',
    lgb.LGBMRegressor(random_state=42, verbose=-1),
    lgb_params,
    X_train_split, y_train_split, X_val, y_val,
    scaled=False
)

optimized_models['LightGBM'] = {'model': best_lgb, 'scaled': False}
tuning_summary.append({
    'Model': 'LightGBM',
    'Best_Val_RMSPE': score_lgb,
    'Best_Params': str(params_lgb)
})


 TUNING LightGBM
Testing 108 combinations...
Fitting 5 folds for each of 108 candidates, totalling 540 fits

 Best Parameters:
   learning_rate: 0.15
   max_depth: -1
   n_estimators: 200
   num_leaves: 70

 Results:
   CV RMSPE: 0.1585
   Val RMSPE: 0.1531
   Val RMSE: 1005.55
   Val MAE:  734.10
   Val R²: 0.9157
   Time: 614.38s
